# IEEE-CIS Fraud Detection

## Notebook 3: Feature Engineering

This notebook creates new features from the cleaned dataset based on patterns
discovered during EDA. Every feature created here has a direct justification
from the exploratory analysis.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

In [2]:
df = pd.read_csv('/kaggle/input/notebooks/aliibtissam/01-data-cleaning/cleaned_data.csv')

print(f'Shape: {df.shape}')

Shape: (590540, 422)


## 1. Time-Based Features

TransactionDT is a time delta in seconds from a fixed reference point.
We extract hour, day of week, and week number to give the model
learnable time-based patterns.

In [3]:
df['transaction_hour'] = (df['TransactionDT'] // 3600) % 24
df['transaction_day']  = (df['TransactionDT'] // (3600 * 24)) % 7
df['transaction_week'] = (df['TransactionDT'] // (3600 * 24 * 7))

print(df[['TransactionDT', 'transaction_hour', 'transaction_day', 'transaction_week']].head())

   TransactionDT  transaction_hour  transaction_day  transaction_week
0          86400                 0                1                 0
1          86401                 0                1                 0
2          86469                 0                1                 0
3          86499                 0                1                 0
4          86506                 0                1                 0


## 2. Transaction Amount Transformation

Raw TransactionAmt has extreme outliers. Log transformation compresses
the scale and produces a more uniform distribution for the model.

In [4]:
df['log_transaction_amt'] = np.log1p(df['TransactionAmt'])

print(df[['TransactionAmt', 'log_transaction_amt']].describe())

       TransactionAmt  log_transaction_amt
count   590540.000000        590540.000000
mean       135.027176             4.382960
std        239.162522             0.937183
min          0.251000             0.223943
25%         43.321000             3.791459
50%         68.769000             4.245190
75%        125.000000             4.836282
max      31937.390000            10.371564


## 3. High Risk Email Flag

EDA showed mail.com has a fraud rate of 18.9% and outlook.com has 9.4%.
We create a binary flag to directly encode this domain knowledge into the model.

In [5]:
high_risk_domains = ['mail.com', 'outlook.com', 'live.com.mx', 'hotmail.com']

df['high_risk_email'] = df['P_emaildomain'].isin(high_risk_domains).astype(int)

print(df['high_risk_email'].value_counts())
print(f'\nFraud rate for high risk email: {df[df["high_risk_email"]==1]["isFraud"].mean()*100:.2f}%')
print(f'Fraud rate for normal email:    {df[df["high_risk_email"]==0]["isFraud"].mean()*100:.2f}%')

high_risk_email
0    538886
1     51654
Name: count, dtype: int64

Fraud rate for high risk email: 5.86%
Fraud rate for normal email:    3.27%


## 4. Card Risk Feature

EDA showed discover cards and credit cards have the highest fraud rates.
We create a binary flag for the highest risk card combination.

In [6]:
df['high_risk_card'] = ((df['card4'] == 'discover') & (df['card6'] == 'credit')).astype(int)

print(df['high_risk_card'].value_counts())
print(f'\nFraud rate for high risk card: {df[df["high_risk_card"]==1]["isFraud"].mean()*100:.2f}%')
print(f'Fraud rate for normal card:    {df[df["high_risk_card"]==0]["isFraud"].mean()*100:.2f}%')

high_risk_card
0    584236
1      6304
Name: count, dtype: int64

Fraud rate for high risk card: 7.93%
Fraud rate for normal card:    3.45%


## 5. Extract Numeric Value from id_34

id_34 contains values like 'match_status:2'. The numeric part carries
the actual signal — we extract it so the model can use it as a number.

In [7]:
# extract the number after the colon in values like 'match_status:2'
df['id_34_numeric'] = df['id_34'].str.extract(r':(\-?\d+)').astype(float)

print(df['id_34_numeric'].value_counts())
print(f'\nMissing values: {df["id_34_numeric"].isnull().sum()}')

id_34_numeric
 2.0    60011
 1.0    17376
 0.0      415
-1.0        3
Name: count, dtype: int64

Missing values: 512735


## 6. Drop High Missing Value V Features

We keep only the top 20 V features by correlation with fraud identified in EDA.
The remaining 319 V features are noise that would slow training and hurt performance.

In [8]:
top_v_features = ['V257', 'V246', 'V244', 'V242', 'V201', 'V200', 
                  'V45', 'V189', 'V86', 'V87', 'V258', 'V294', 
                  'V167', 'V187', 'V188', 'V307', 'V308', 'V310',
                  'V156', 'V130']

all_v_features = [col for col in df.columns if col.startswith('V')]
v_features_to_drop = [col for col in all_v_features if col not in top_v_features]

df.drop(columns=v_features_to_drop, inplace=True)

print(f'V features kept  : {len(top_v_features)}')
print(f'V features dropped: {len(v_features_to_drop)}')
print(f'Remaining columns : {df.shape[1]}')

V features kept  : 20
V features dropped: 319
Remaining columns : 110


In [9]:
df.drop(columns=['TransactionDT', 'id_34'], inplace=True)

print(f'Final shape: {df.shape}')

Final shape: (590540, 108)


## Summary

| Feature Created | Source Column | Reason |
|---|---|---|
| transaction_hour | TransactionDT | Fraud fluctuates by hour of day |
| transaction_day | TransactionDT | Fraud fluctuates by day of week |
| transaction_week | TransactionDT | Captures longer term time patterns |
| log_transaction_amt | TransactionAmt | Compresses skewed distribution |
| high_risk_email | P_emaildomain | mail.com has 5x average fraud rate |
| high_risk_card | card4, card6 | Discover credit has 2x average fraud rate |
| id_34_numeric | id_34 | Extracts usable number from string |

Columns reduced from 422 to 108 by dropping 319 low-signal V features
and 2 columns replaced by engineered features.

In [10]:
df.to_csv('feature_engineered_data.csv', index=False)

print(f'Saved: feature_engineered_data.csv')
print(f'Shape: {df.shape}')

Saved: feature_engineered_data.csv
Shape: (590540, 108)
